# 01: Data Exploration (Geological Drill Cores)

This notebook verifies the structural integrity of the raw ENVI hyperspectral data (`FENIX.dat`) and the high-resolution Mineral Liberation Analysis (`MLA_HSI.dat`) ground truth maps.

## Prerequisites
1. You must have manually downloaded the **HZDR RODARE Upscaling Mineralogy Benchmark** dataset.
2. The raw ENVI `.dat` and `.hdr` files along with `AbundanceMapping.xlsx` must be uploaded to your Google Drive in the folder: `MyDrive/hsi-experiments/data/raw/MLA`


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
data_dir = '/content/drive/MyDrive/hsi-experiments/data/raw/MLA'

# Verify the data exists
if not os.path.exists(data_dir):
    print(f"ERROR: Could not find {data_dir}. Please upload the dataset first.")
else:
    print(f"Successfully found data directory: {data_dir}")


In [ ]:
# Clone the repository so we can use the custom src.data.download module
!git clone https://github.com/suva1998/hsi-lithology-showcase.git
%cd hsi-lithology-showcase
!pip install -e .


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import spectral.io.envi as envi
import pandas as pd
import os

# ---------------------------------------------------------------------------
# Helper: load a single drill core sample directly from its .hyc directory
# ---------------------------------------------------------------------------
def load_single_core(hyc_dir):
    """Load FENIX HSI cube, MLA abundance labels, and mask for one core."""
    fenix_hdr = os.path.join(hyc_dir, 'FENIX.hdr')
    fenix_dat = os.path.join(hyc_dir, 'FENIX.dat')
    mla_hdr   = os.path.join(hyc_dir, 'MLA_HSI.hdr')
    mla_dat   = os.path.join(hyc_dir, 'MLA_HSI.dat')

    # Load HSI cube
    img = envi.open(fenix_hdr, fenix_dat)
    hsi = np.nan_to_num(img.load().astype(np.float32))

    # Read reflectance scale factor from header
    scale = float(img.metadata.get('reflectance scale factor', 1.0))
    if scale > 1.0:
        hsi = hsi * scale  # restore to [0, 1] reflectance

    # Load MLA abundance map
    mla_img = envi.open(mla_hdr, mla_dat)
    mla_cube = mla_img.load()
    band_names = [b.strip() for b in mla_img.metadata.get('band names', [])]

    # Dominant mineral per pixel
    dominant_band = np.argmax(mla_cube, axis=-1)
    max_abundance = np.max(mla_cube, axis=-1)

    # Apply mask if available
    mask_hdr = os.path.join(hyc_dir, 'mask.hdr')
    mask_dat = os.path.join(hyc_dir, 'mask.dat')
    if os.path.exists(mask_hdr):
        mask = envi.open(mask_hdr, mask_dat).load().squeeze()
        min_h = min(hsi.shape[0], mask.shape[0])
        min_w = min(hsi.shape[1], mask.shape[1])
        hsi = hsi[:min_h, :min_w, :]
        dominant_band = dominant_band[:min_h, :min_w]
        max_abundance = max_abundance[:min_h, :min_w]
        dominant_band[mask[:min_h, :min_w] == 0] = -1  # mark as background

    # Mark pixels with zero total abundance as background
    dominant_band[max_abundance == 0] = -1

    return hsi, dominant_band, band_names


def make_rgb(hsi, r_band=78, g_band=51, b_band=25):
    """Create a percentile-stretched visible-light RGB composite."""
    rgb = np.stack([hsi[:,:,r_band], hsi[:,:,g_band], hsi[:,:,b_band]], axis=2)
    for c in range(3):
        ch = rgb[:,:,c]
        valid = ch[ch > 0]
        if len(valid) > 0:
            lo, hi = np.percentile(valid, [2, 98])
            ch = np.clip(ch, lo, hi)
            ch = (ch - lo) / (hi - lo + 1e-8)
        rgb[:,:,c] = ch
    return rgb

print('Helper functions loaded.')


In [ ]:
# ---------------------------------------------------------------------------
# Visualize 2 individual Stonepark drill core samples
# ---------------------------------------------------------------------------

# Pick two specific cores with different characteristics
stonepark_dir = os.path.join(data_dir, 'MLA.shed', 'Stonepark.hyc')
sample_names = ['bG11A1', 'bTC3616']

fig, axes = plt.subplots(2, 2, figsize=(10, 14))

for row, name in enumerate(sample_names):
    hyc_path = os.path.join(stonepark_dir, f'{name}.hyc')
    print(f'Loading {name}...')
    hsi, labels, mineral_names = load_single_core(hyc_path)
    print(f'  HSI shape: {hsi.shape}  |  Minerals: {len(mineral_names)}')
    print(f'  Value range: [{hsi.min():.4f}, {hsi.max():.4f}]')

    # --- RGB composite ---
    rgb = make_rgb(hsi)
    axes[row, 0].imshow(rgb, interpolation='nearest')
    axes[row, 0].set_title(f'{name} — Visible RGB (644/551/463 nm)', fontsize=11)
    axes[row, 0].axis('off')

    # --- MLA mineral map ---
    n_minerals = len(mineral_names)
    cmap = matplotlib.colormaps.get_cmap('tab20').resampled(n_minerals + 1)
    # Shift labels so background (-1) becomes 0, minerals become 1..N
    display_labels = labels.copy() + 1
    all_names = ['Background'] + mineral_names

    im = axes[row, 1].imshow(display_labels, cmap=cmap,
                              vmin=0, vmax=len(all_names)-1,
                              interpolation='nearest')
    axes[row, 1].set_title(f'{name} — MLA Dominant Mineral', fontsize=11)
    axes[row, 1].axis('off')

    # Colorbar legend
    cbar = plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)
    cbar.set_ticks(np.arange(len(all_names)))
    cbar.set_ticklabels(all_names, fontsize=7)

plt.suptitle('HZDR RODARE Drill Core Samples (Stonepark)', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Print dataset statistics across available drill sites
# ---------------------------------------------------------------------------
shed_dir = os.path.join(data_dir, 'MLA.shed')

for site_name in sorted(os.listdir(shed_dir)):
    site_path = os.path.join(shed_dir, site_name)
    if not site_name.endswith('.hyc') or not os.path.isdir(site_path):
        continue

    # Count valid samples in this site
    hyc_dirs = [d for d in os.listdir(site_path)
                if d.endswith('.hyc') and os.path.isdir(os.path.join(site_path, d))]
    # Check one sample for dimensions
    if hyc_dirs:
        sample_dir = os.path.join(site_path, hyc_dirs[0])
        fenix_hdr = os.path.join(sample_dir, 'FENIX.hdr')
        if os.path.exists(fenix_hdr):
            img = envi.open(fenix_hdr)
            scale = img.metadata.get('reflectance scale factor', '1.0')
            mla_hdr = os.path.join(sample_dir, 'MLA_HSI.hdr')
            n_mla = 0
            if os.path.exists(mla_hdr):
                mla = envi.open(mla_hdr)
                n_mla = mla.nbands
            print(f'{site_name:20s}  samples={len(hyc_dirs):3d}  '
                  f'shape=({img.nrows}x{img.ncols}x{img.nbands})  '
                  f'MLA_bands={n_mla}  scale={scale}')
